In [3]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

# Example: each star is a dict with pos, mass, velocity
data = {
    "galaxy_id": [1, 2],
    "stars": [
        [
            {"pos": [1.2, 3.4, 5.1], "mass": 1.5, "velocity": [100, 10, 0]},
            {"pos": [2.2, 4.5, 6.7], "mass": 0.9, "velocity": [-50, 20, 5]},
        ],
        [
            {"pos": [7.7, 8.8, 9.9], "mass": 2.0, "velocity": [200, -10, 15]},
        ],
    ],
    "short": [101, 102],
}
df = pd.DataFrame(data)

# Define arrow types for the schema
star_struct = pa.struct(
    [
        ("pos", pa.list_(pa.float32(), list_size=3)),
        ("mass", pa.float32()),
        ("velocity", pa.list_(pa.float32(), list_size=3)),
        ("short", pa.int8()),
    ]
)
stars_type = pa.list_(star_struct)

schema = pa.schema([("galaxy_id", pa.int64()), ("stars", stars_type)])

table = pa.Table.from_pandas(df, schema=schema, preserve_index=False)
pq.write_table(table, "galaxies_with_star_properties.parquet")

In [4]:
import pyarrow.dataset as ds

dataset = ds.dataset("galaxies_with_star_properties.parquet", format="parquet")
print("Number of rows:", dataset.count_rows())
print("Dataset schema:\n", dataset.schema)

Number of rows: 2
Dataset schema:
 galaxy_id: int64
stars: list<element: struct<pos: fixed_size_list<element: float>[3], mass: float, velocity: fixed_size_list<element: float>[3], short: int8>>
  child 0, element: struct<pos: fixed_size_list<element: float>[3], mass: float, velocity: fixed_size_list<element: float>[3], short: int8>
      child 0, pos: fixed_size_list<element: float>[3]
          child 0, element: float
      child 1, mass: float
      child 2, velocity: fixed_size_list<element: float>[3]
          child 0, element: float
      child 3, short: int8
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 305
